# Diagnosis: Where could reranking help?

Find labeled questions with Recall@30 greater than Recall@5 using the rebuilt Qwen retrieval. Counts and example ranks are observed anew.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.

**Study scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions, the effervescent diagnostic, and a separate IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** Experiment 1 (Baseline) --> **this notebook Diagnosis** --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


## 1. Load the embedding-only records

**What:** load the frozen questions and annotated support IDs, then embed each question and retrieve its 30 candidates.

**Why:** make the opportunity concrete and inspectable using fresh question embeddings and the fixed passage index.

Use the project environment and prepared inputs described in [setup.md](../setup.md). All calculations below use embedding ranks. No reranker ordering, answer text, SQuAD or RAGAS score determines membership.

In [1]:
STUDY = 'diagnosis'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")
view.show('questions')


,Setting,Value
0,Dataset,MS MARCO v2.1 validation
1,Embedding model,qwen/qwen3-embedding-4b
2,Embedding dimensions,2560
3,Retrieval,FAISS IndexIVFFlat; L2-normalized cosine search
4,Candidate / context budgets,30 / 5 passages
5,Passage limit,"1,800 characters in the upstream retrieval setup"
6,Records available here,1085
7,Model calls,Fresh retrieval: inspect runtime request logs


In [ ]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

retrieval_run = tutorial.retrieve(embed_api)
if retrieval_run is not None:
    display(retrieval_run)


## 2. Diagnose the frozen validation subset

Compare annotated-support coverage at five versus thirty passages across the **15,585-question frozen subset**. Of these, **10,242** have known support labels. The subset summary reports Recall@5 **0.8918**, Recall@30 **0.9910**, and **1,085 opportunity questions**.

Recall@k is the fraction of a question's annotated supporting passages found in its first k retrieved passages. These are known-label measurements, not exhaustive relevance judgments. The table below summarizes the subset; it does not estimate population averages from opportunity rows.

**Scope:** the table's “full-validation” heading refers here to this frozen subset, not the full MS MARCO validation split.


In [2]:
diagnosis_run = tutorial.diagnose()
if diagnosis_run is not None:
    display(diagnosis_run)
view.show('retrieval_summary')


,Previously reported full-validation result,Value
0,Validation questions,"15,585"
,Previously reported full-validation result,Value
1,Questions with support annotations,"10,242"
2,Reported mean Recall@5,0.8918
3,Reported mean Recall@30,0.9910
4,Reported mean recall gap,0.0992
5,All annotated support in top five,"9,062"
6,All annotated support in top thirty,"10,139"
7,Additional support in ranks 6–30,"1,085"
8,Some support absent from top thirty,103


## 3. Define the opportunity before testing reranking

**What:** select labeled questions where **Recall@30 > Recall@5**.

**Why:** some annotated evidence is already retrieved at ranks 6–30, but would be left out of a five-passage context. Reordering could help; it has not yet been shown to help.

| Embedding-only situation | What it tells us |
|---|---|
| Recall@30 > Recall@5 | Evidence below the cutoff is recoverable by selection or reordering. |
| Recall@30 = Recall@5 = 1 | All annotated support is already in the top five. |
| Recall@30 = Recall@5 < 1 | Missing annotated support is outside the top thirty; reordering cannot recover those missing IDs. |

Query IDs are stable identifiers, not spreadsheet row numbers. `query_row` is the original dataset row position. The rule depends on support annotations and embedding ranks, not on a generated answer.

In [3]:
if MODE != "plan":
    retrieval_metrics = view.retrieval()
    display(retrieval_metrics.groupby("opportunity").size().rename("questions"))


,Denominator,Questions,opportunity_questions,opportunity_share_pct
0,All validation questions,15585,1085,6.96
1,Questions with annotations,10242,1085,10.59


## 4. Show the actual opportunity questions

**What:** inspect questions with no annotated support in the top five, and questions missing some complementary support.

**Why:** explain what the selection rule means for real queries rather than stop at an aggregate.

These are embedding-only categories. They are not the later gain/unchanged/loss groups. The displayed examples are ordered deterministically by original dataset row; they are illustrations, not representative random samples. The complete selected question table is `data/opportunity_records.csv`.

In [4]:
view.show('opportunities')


,questions,mean_recall_at_5,mean_recall_at_30,mean_recall_gap
embedding_situation,,,,
No annotated support in top five,957,0.0000,0.9969,0.9969
Some support in top five; additional support below cutoff,128,0.5109,0.9954,0.4845



No annotated support in top five


,query_row,query_id,question,recall_at_5,recall_at_30,recall_gap
2,38,1090278,symptoms of air pollution exposure,0.0,1.0,1.0
3,54,174249,does xpress bet charge to deposit money in you...,0.0,1.0,1.0
4,73,180613,english is the official state language in some...,0.0,1.0,1.0
5,84,330883,how much would graduate marketers make?,0.0,1.0,1.0
6,86,79604,can you set google chrome as your default web ...,0.0,1.0,1.0



Some support in top five; additional support below cutoff


,query_row,query_id,question,recall_at_5,recall_at_30,recall_gap
0,1,1102431,why did rachel carson write an obligation to e...,0.500000,1.0,0.500000
1,26,114414,"current weather in volcano, ca",0.666667,1.0,0.333333
11,136,236184,how is alkali made,0.500000,1.0,0.500000
14,179,190280,foods to firm up stool,0.500000,1.0,0.500000
15,181,178627,effects of detox juice cleanse,0.500000,1.0,0.500000


## 5. Verify coverage directly from passage IDs

**What:** join annotated support IDs to the embedding-ranked candidate IDs, then count support in the first five and thirty.

**Why:** verify the recall columns from evidence membership rather than trust copied scores.

The denominator includes **all** annotated support for each question. A support passage outside the top thirty has a missing candidate rank and counts as a miss. Finding one of two annotated passages gives recall 0.5. This audit verifies the bundled selected records, not all 15,585 questions in the frozen subset.


In [5]:
# Recall is recomputed directly from gold IDs and this run's candidate ranks.
view.show('retrieval')
view.show('audit')


Verified support-ID coverage for all 1,085 opportunity questions.


,query_id,question,annotated_passages,support_in_top5,support_in_top30,recoverable_support,first_support_rank
0,1102431,why did rachel carson write an obligation to e...,2,1,2,1,1.0
,query_id,question,annotated_passages,support_in_top5,support_in_top30,recoverable_support,first_support_rank
1,114414,"current weather in volcano, ca",3,2,3,1,4.0
2,1090278,symptoms of air pollution exposure,1,0,1,1,7.0
3,174249,does xpress bet charge to deposit money in you...,1,0,1,1,7.0
4,180613,english is the official state language in some...,1,0,1,1,15.0
5,330883,how much would graduate marketers make?,1,0,1,1,12.0
6,79604,can you set google chrome as your default web ...,1,0,1,1,6.0
7,1101279,do physicians pay for insurance from their sal...,1,0,1,1,6.0
8,96252,complex variable functions,1,0,1,1,6.0


## 6. Inspect one question’s cutoff

Inspect an observed opportunity in the new Qwen retrieval. Its annotated support ranks are recomputed.


In [6]:
view.show('contexts')


Query 1102431: why did rachel carson write an obligation to endure


,Embedding-only measurement,Value
0,Annotated-support Recall@5,0.5
1,Annotated-support Recall@30,1.0
2,First annotated-support rank,1.0


,query_id,passage_id,embedding_rank,annotated_support,would_enter_top5_context
0,1102431,p_a7d7c1627719a2739347,1,True,True
1,1102431,p_798501edc35d7af8b254,2,False,True
2,1102431,p_4b697b2575e680939e1e,3,False,True
3,1102431,p_aca5d9930a137042d66a,4,False,True
4,1102431,p_9291f65e1a1abad7633a,5,False,True
8,1102431,p_df702fcc14ec0419b842,9,True,False



Fresh saved annotated-support excerpt: p_a7d7c1627719a2739347, embedding rank 1
The Obligation to Endure by Rachel Carson Rachel Carson's essay on The Obligation to Endure, is a very convincing argument about the harmful uses of chemicals, pesticides, herbicides, and fertilizers on the environment.

Fresh saved annotated-support excerpt: p_df702fcc14ec0419b842, embedding rank 9
Carson believes that as man tries to eliminate unwanted insects and weeds, however he is actually causing more problems by polluting the environment with, for example, DDT and harming living things. Carson adds that the intensification of agriculture is causing other major problems, like newly developed or created insects and diseases.


## 7. Hand a fixed sample to targeted reranking

The diagnosis identifies **1,085** opportunity questions using Recall@30 > Recall@5. Experiment 2 samples **500** opportunities and **200** no-gap controls with the configured seed before reranking. It reuses each selected question's exact thirty candidates. The separate effervescent diagnostic does not enter those group averages.


In [ ]:
view.show('retrieval_summary')
tutorial.show()


## Optional appendix: reproduce selection from the upstream subset table

Supply the original upstream CSV with `query_id`, `has_known_support_labels`, `recall_at_5` and `recall_at_30`. An optional `question` column makes the displayed records easier to inspect. Set `FULL_VALIDATION_METRICS_PATH` privately, then enable the switch.

This is an optional local check, separate from the live retrieval workflow. It verifies the frozen-subset count and exact opportunity-ID set. It does not sample controls or request a model. The provenance file retains the original upstream file hash for comparison.


In [7]:
# Local cross-check of the opportunity rule; no additional model calls.
if MODE != "plan":
    checked = view.retrieval()
    opportunity_ids = checked.loc[checked.opportunity, "query_id"].tolist()
    assert len(opportunity_ids) == len(set(opportunity_ids))
    display({"opportunity_count": len(opportunity_ids), "rule": "Recall@30 > Recall@5"})


Full-validation check disabled: only the bundled opportunity records were recalculated.


### Reproducibility

The runtime data directory contains question-level recall, ranked candidate IDs, support annotations, provenance, and the subset summary. Some filenames and original output headings retain “full_validation” for compatibility. Their scope is the frozen 15,585-question subset.
